# Use a released SeLMRoute CatBoost router offline

This notebook demonstrates the released deployment router **without** running a semantic model. It uses semantic features already published for an LLMRouterBench query. The default backend is Laya.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

backend = 'laya'   # change to 'jev' to use the JEV-trained checkpoint
samples = pd.read_csv(ROOT/'data/performance/samples.csv', dtype={'sample_id': str})
samples[['sample_id','dataset','domain','query']].head()

,sample_id,dataset,domain,query
0,aime:hybrid:10:7d2b9ecfeeda,aime,math,Alice chooses a set $A$ of positive integers. ...
1,aime:hybrid:11:f2fe46bf765d,aime,math,Find the largest possible real part of \[(75+1...
2,aime:hybrid:12:73df31e8148e,aime,math,Find the number of ways to place a digit in ea...
3,aime:hybrid:13:60f7c1749379,aime,math,Every morning Aya goes for a $9$-kilometer-lon...
4,aime:hybrid:14:10c80ba48d18,aime,math,Let $N$ be the greatest four-digit positive in...


Choose any published query. The example below uses the first sample only to keep the notebook deterministic.

In [2]:
sample_id = str(samples.iloc[0].sample_id)
query = str(samples.iloc[0].query)

print("Sample:", sample_id)
print("\nQuery:")
print(query)

Sample: aime:hybrid:10:7d2b9ecfeeda

Query:
Alice chooses a set $A$ of positive integers. Then Bob lists all finite nonempty sets $B$ of positive integers with the property that the maximum element of $B$ belongs to $A$. Bob's list has 2024 sets. Find the sum of the elements of A.


In [3]:
from selmroute.inference import route_published_sample

result = route_published_sample(
    sample_id,
    backend=backend,
    root=ROOT,
    top_k=5,
    verbose=True,
)

result

{'sample_id': 'aime:hybrid:10:7d2b9ecfeeda',
 'query': "Alice chooses a set $A$ of positive integers. Then Bob lists all finite nonempty sets $B$ of positive integers with the property that the maximum element of $B$ belongs to $A$. Bob's list has 2024 sets. Find the sum of the elements of A.",
 'dataset': 'aime',
 'domain': 'math',
 'backend': 'laya',
 'selected_model': 'DeepSeek-R1-0528-Qwen3-8B',
 'ranking': [{'rank': 1,
   'model': 'DeepSeek-R1-0528-Qwen3-8B',
   'predicted_score': 0.8380041322243323,
   'observed_score': 1.0},
  {'rank': 2,
   'model': 'Qwen3-8B',
   'predicted_score': 0.8360032787529106,
   'observed_score': 1.0},
  {'rank': 3,
   'model': 'MiniCPM4.1-8B',
   'predicted_score': 0.8137028692026964,
   'observed_score': 1.0},
  {'rank': 4,
   'model': 'NVIDIA-Nemotron-Nano-9B-v2',
   'predicted_score': 0.8002405765261598,
   'observed_score': 1.0},
  {'rank': 5,
   'model': 'DeepSeek-R1-Distill-Qwen-7B',
   'predicted_score': 0.7747866209401432,
   'observed_score'

`predicted_score` is the CatBoost estimate for each candidate. `observed_score` is shown only because the selected query belongs to the published benchmark. It is never supplied to the router.

In [4]:
pd.DataFrame(result['ranking'])

,rank,model,predicted_score,observed_score
0,1,DeepSeek-R1-0528-Qwen3-8B,0.838004,1.0
1,2,Qwen3-8B,0.836003,1.0
2,3,MiniCPM4.1-8B,0.813703,1.0
3,4,NVIDIA-Nemotron-Nano-9B-v2,0.800241,1.0
4,5,DeepSeek-R1-Distill-Qwen-7B,0.774787,1.0
